In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "OCQ_D.xpt"
csv_file = DATA_DIR / "OCQ_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched source copy for final fidelity checking
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 OCQ_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "OCD150",
    "OCQ180",
    "OCQ210",
    "OCD231",
    "OCD241",
    "OCQ260",
    "OCQ265",
    "OCD270",
    "OCQ290G",
    "OCQ290Q",
    "OCQ380",
    "OCD390G",
    "OCD392",
    "OCD395"
]


if len(df) != 6139:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 15:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "OCQ_D columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: OCQ_D structure = "
    "6,139 rows x 15 columns."
)


# ============================================================
# 4. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. EXPECTED VALIDATED TINY COUNTS
# ============================================================

EXPECTED_TINY_COUNTS = {

    "OCD270": 104,

    "OCQ290Q": 37,

    "OCD395": 30
}


EXPECTED_TINY_TOTAL = 171


# ============================================================
# 6. VALIDATE TINY ARTIFACT LOCATIONS
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)


expected_tiny_columns = set(
    EXPECTED_TINY_COUNTS.keys()
)


unexpected_columns = (
    actual_tiny_columns
    -
    expected_tiny_columns
)


if unexpected_columns:

    raise ValueError(
        "STOP: Tiny values found in unexpected "
        "OCQ_D variables:\n"
        + str(sorted(unexpected_columns))
    )


missing_expected_columns = (
    expected_tiny_columns
    -
    actual_tiny_columns
)


if missing_expected_columns:

    raise ValueError(
        "STOP: Expected tiny-value variables missing:\n"
        + str(sorted(missing_expected_columns))
    )


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        tiny_counts[col]
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


if int(
    tiny_counts.sum()
) != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny-value count."
    )


print(
    "\nPASS: Exactly 171 validated "
    "tiny artifacts found."
)


# ============================================================
# 7. RESTORE VALIDATED ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    count = int(
        mask.sum()
    )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{count:,} tiny -> 0"
    )


# ============================================================
# 8. VERIFY RESTORED ZERO COUNTS
# ============================================================

print("\n--- RESTORED ZERO COUNTS ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<10}: "
        f"{zero_count:,}"
    )


    if zero_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} zeros, "
            f"found {zero_count:,}."
        )


print(
    "PASS: All 171 validated zeros restored."
)


# ============================================================
# 9. VERIFY NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny-value artifact remains."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 10. CONFIRM NO GENUINE DECIMAL VARIABLES
# ============================================================

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_failures = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    print(
        f"{col:<10}: "
        f"fractional values = {count:,}"
    )


    if count != 0:

        fractional_failures.append(
            col
        )


if fractional_failures:

    raise ValueError(
        "STOP: Genuine decimal values found in:\n"
        + str(fractional_failures)
    )


print(
    "PASS: OCQ_D contains no genuine decimal variables."
)


# ============================================================
# 11. SEQN VALIDATION
# ============================================================

print("\n--- SEQN CHECK ---")


seqn_values = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn_values,
        np.round(seqn_values),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


seqn_missing = int(
    df["SEQN"]
    .isna()
    .sum()
)


seqn_duplicates = int(
    df["SEQN"]
    .duplicated()
    .sum()
)


seqn_min = df["SEQN"].min()
seqn_max = df["SEQN"].max()


print(
    "Missing:",
    seqn_missing
)

print(
    "Duplicates:",
    seqn_duplicates
)

print(
    "Minimum:",
    seqn_min
)

print(
    "Maximum:",
    seqn_max
)


if seqn_missing != 0:
    raise ValueError(
        "Unexpected missing SEQN."
    )


if seqn_duplicates != 0:
    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn_min != 31130:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn_min}"
    )


if seqn_max != 41474:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn_max}"
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 12. CDC CATEGORICAL FREQUENCIES
# ============================================================

EXPECTED_FREQUENCIES = {

    "OCD150": {
        "counts": {
            1: 3208,
            2: 149,
            3: 187,
            4: 2593,
            7: 2
        },
        "missing": 0
    },

    "OCQ210": {
        "counts": {
            1: 316,
            2: 685,
            7: 1,
            9: 3
        },
        "missing": 5134
    },

    "OCQ260": {
        "counts": {
            1: 2550,
            2: 88,
            3: 197,
            4: 205,
            5: 285,
            6: 15,
            77: 1,
            99: 16
        },
        "missing": 2782
    },

    "OCQ265": {
        "counts": {
            1: 2297,
            2: 253,
            3: 145,
            4: 330,
            5: 330,
            7: 1,
            9: 1
        },
        "missing": 2782
    },

    "OCQ290G": {
        "counts": {
            1: 717,
            2: 2639,
            7: 1
        },
        "missing": 2782
    },

    "OCQ380": {
        "counts": {
            1: 485,
            2: 470,
            3: 972,
            4: 206,
            5: 28,
            6: 213,
            7: 216,
            77: 1
        },
        "missing": 3548
    },

    "OCD390G": {
        "counts": {
            1: 3388,
            2: 2191,
            3: 47,
            4: 499,
            7: 11,
            9: 2
        },
        "missing": 1
    }
}


# ============================================================
# 13. VALIDATE CDC CATEGORICAL FREQUENCIES
# ============================================================

print("\n--- CDC CATEGORICAL FREQUENCY CHECKS ---")


for col, spec in EXPECTED_FREQUENCIES.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v
        in actual_counts.items()
    }


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_counts != spec["counts"]:

        raise ValueError(
            f"{col}: CDC frequency mismatch.\n"
            f"Expected: {spec['counts']}\n"
            f"Actual:   {actual_counts}"
        )


    if actual_missing != spec["missing"]:

        raise ValueError(
            f"{col}: missing count mismatch.\n"
            f"Expected: {spec['missing']:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {col:<8} "
        f"valid={sum(actual_counts.values()):,} "
        f"missing={actual_missing:,}"
    )


print(
    "PASS: Categorical frequencies match CDC."
)


# ============================================================
# 14. CDC CHECK - OCQ180
# ============================================================

#
# Hours worked last week.
#
# CDC:
#
# 1-125   = 3,205
# 77777   = 2
# 99999   = 1
# Missing = 2,931
#

s = df["OCQ180"]


if not (
    int(
        s.between(
            1,
            125
        ).sum()
    ) == 3205

    and

    int(
        s.eq(77777)
        .sum()
    ) == 2

    and

    int(
        s.eq(99999)
        .sum()
    ) == 1

    and

    int(
        s.isna()
        .sum()
    ) == 2931
):

    raise ValueError(
        "OCQ180 differs from CDC."
    )


print(
    "\nPASS: OCQ180 matches CDC."
)


# ============================================================
# 15. CDC CHECK - OCD231 / OCD241
# ============================================================

#
# Current industry and occupation group codes.
#

for col in [
    "OCD231",
    "OCD241"
]:

    s = df[col]


    if not (
        int(
            s.notna()
            .sum()
        ) == 3357

        and

        int(
            s.isna()
            .sum()
        ) == 2782

        and

        s.min() >= 1

        and

        s.max() <= 99
    ):

        raise ValueError(
            f"{col} differs from CDC."
        )


    print(
        f"PASS: {col} matches CDC "
        "(3,357 valid / 2,782 missing)."
    )


# ============================================================
# 16. CDC CHECK - OCD270
# ============================================================

#
# Months in current main job.
#
# CDC:
#
# 1-720   = 3,251
# 0       = 104   Less than 1 month
# 77777   = 1
# 99999   = 1
# Missing = 2,782
#

s = df["OCD270"]


if not (
    int(
        s.between(
            1,
            720
        ).sum()
    ) == 3251

    and

    int(
        s.eq(0)
        .sum()
    ) == 104

    and

    int(
        s.eq(77777)
        .sum()
    ) == 1

    and

    int(
        s.eq(99999)
        .sum()
    ) == 1

    and

    int(
        s.isna()
        .sum()
    ) == 2782
):

    raise ValueError(
        "OCD270 differs from CDC."
    )


print(
    "PASS: OCD270 matches CDC."
)

print(
    "      Zero / Less than 1 month = 104"
)


# ============================================================
# 17. CDC CHECK - OCQ290Q
# ============================================================

#
# Hours/day smelling tobacco smoke.
#
# CDC specifically says:
#
# IF NUMBER OF HOURS IS LESS THAN ONE,
# ENTER ZERO.
#
# 0-20    = 714
# 9999    = 3
# Missing = 5,422
#
# Actual zeros = 37
#

s = df["OCQ290Q"]


if not (
    int(
        s.between(
            0,
            20
        ).sum()
    ) == 714

    and

    int(
        s.eq(0)
        .sum()
    ) == 37

    and

    int(
        s.eq(9999)
        .sum()
    ) == 3

    and

    int(
        s.isna()
        .sum()
    ) == 5422
):

    raise ValueError(
        "OCQ290Q differs from CDC/source."
    )


print(
    "PASS: OCQ290Q matches CDC/source."
)

print(
    "      Zero / Less than 1 hour = 37"
)


# ============================================================
# 18. CDC CHECK - OCD392
# ============================================================

#
# Occupation group code for longest job.
#

s = df["OCD392"]


if not (
    int(
        s.notna()
        .sum()
    ) == 3435

    and

    int(
        s.isna()
        .sum()
    ) == 2704

    and

    s.min() >= 1

    and

    s.max() <= 98
):

    raise ValueError(
        "OCD392 differs from CDC."
    )


print(
    "PASS: OCD392 matches CDC "
    "(3,435 valid / 2,704 missing)."
)


# ============================================================
# 19. CDC CHECK - OCD395
# ============================================================

#
# Duration of longest job in months.
#
# CDC:
#
# 1-720   = 3,396
# 0       = 30  Less than 1 month
# 77777   = 2
# 99999   = 7
# Missing = 2,704
#

s = df["OCD395"]


if not (
    int(
        s.between(
            1,
            720
        ).sum()
    ) == 3396

    and

    int(
        s.eq(0)
        .sum()
    ) == 30

    and

    int(
        s.eq(77777)
        .sum()
    ) == 2

    and

    int(
        s.eq(99999)
        .sum()
    ) == 7

    and

    int(
        s.isna()
        .sum()
    ) == 2704
):

    raise ValueError(
        "OCD395 differs from CDC."
    )


print(
    "PASS: OCD395 matches CDC."
)

print(
    "      Zero / Less than 1 month = 30"
)


# ============================================================
# 20. EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# Expected source =
#
# original XPT
# +
# ONLY:
#
# OCD270   tiny -> 0
# OCQ290Q  tiny -> 0
# OCD395   tiny -> 0
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 21. CONFIRM ALL FIELDS ARE WHOLE NUMBERS
# ============================================================

for col in df.columns:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if int(
        fractional.sum()
    ) != 0:

        raise ValueError(
            f"STOP: {col} contains genuine decimal values."
        )


print(
    "\nPASS: All 15 OCQ_D fields "
    "are exact whole-number fields."
)


# ============================================================
# 22. CLEAN SEQN + QUESTIONNAIRE FORMATTING
# ============================================================

#
# Examples:
#
# SEQN
# 31130.0 -> 31130
#
# OCQ180
# 40.0 -> 40
#
# OCD270
# tiny -> 0
#
# OCQ290Q
# tiny -> 0
#
# OCD395
# tiny -> 0
#
# Missing
# NaN -> blank
#

for col in df.columns:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: SEQN and questionnaire fields "
    "converted to clean integer formatting."
)


# ============================================================
# 23. VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


pre_export_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: Values match expected XPT "
    "after ONLY the three validated zero restorations."
)


# ============================================================
# 24. EXPORT CSV
# ============================================================

#
# NO:
#
# float_format
# round()
# recalculation
#
# Missing SAS numeric values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 25. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 26. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 27. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


numeric_failures = []
total_numeric_differences = 0


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Total numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 15 CSV columns numerically "
    "match expected XPT values."
)


# ============================================================
# 28. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    diff = int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    missing_mismatches += diff


    if diff > 0:

        missing_failures.append(
            (
                col,
                diff
            )
        )


print(
    "\nMissing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing-value positions changed:\n"
        + str(missing_failures)
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 29. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


if len(raw_rows) != 6139:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 30. STRICT RAW FORMAT FIDELITY
# ============================================================

#
# All OCQ_D variables are whole-number
# identifiers/codes/count fields.
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in expected_df.columns:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "Examples:"
    )


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "OCQ_D raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV formatting matches "
    "whole-number questionnaire representation."
)


# ============================================================
# 31. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nValues ending '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        "Examples:",
        dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 32. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 33. FINAL TINY-ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 34. FINAL ZERO CHECK
# ============================================================

print("\n--- FINAL CSV ZERO CHECK ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<10}: "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: CSV zero count changed."
        )


print(
    "PASS: Final CSV zero restorations remain correct."
)


# ============================================================
# 35. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("OCQ_D CONVERSION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "\nValidated tiny -> zero:"
)

print(
    "OCD270:",
    104
)

print(
    "OCQ290Q:",
    37
)

print(
    "OCD395:",
    30
)

print(
    "TOTAL:",
    EXPECTED_TINY_TOTAL
)

print(
    "\nGenuine decimal variables:",
    0
)

print(
    "SEQN/code/count formatting:",
    "integer"
)

print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Values ending .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)

print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "OCQ_D_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2005-2006 OCQ_D XPT -> CSV
Rows: 6,139
Columns: 15
PASS: OCQ_D structure = 6,139 rows x 15 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 171

Tiny values by variable:
OCD270     104
OCQ290Q     37
OCD395      30

PASS: Exactly 171 validated tiny artifacts found.

--- ZERO RESTORATION ---
OCD270    : 104 tiny -> 0
OCQ290Q   : 37 tiny -> 0
OCD395    : 30 tiny -> 0

--- RESTORED ZERO COUNTS ---
OCD270    : 104
OCQ290Q   : 37
OCD395    : 30
PASS: All 171 validated zeros restored.

Remaining tiny artifacts: 0
PASS: No tiny-value artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN      : fractional values = 0
OCD150    : fractional values = 0
OCQ180    : fractional values = 0
OCQ210    : fractional values = 0
OCD231    : fractional values = 0
OCD241    : fractional values = 0
OCQ260    : fractional values = 0
OCQ265    : fractional values = 0
OCD270    : fractional values = 0
OCQ290G   : fractional values = 0
OCQ290Q   : fractional value